# 🧪 PT-W2-D2 概念实验：Relationship 抽取——ADR-006 四类关系

> 配套阅读：`PT-W2-D2-Relationship抽取.md`
> 用邻接表构建关系图，BFS 遍历回答"A101 为什么不能出租？"

## 第 1 格：四类关系——ADR-006 §3

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

class RelCategory(Enum):
    IDENTITY_REF = "身份引用"
    STRUCTURAL = "结构组成"
    HIERARCHICAL = "层级包含"
    LIFECYCLE_EFFECT = "生命周期迁移影响"

@dataclass(frozen=True)
class Relation:
    predicate: str
    source: str
    target: str
    category: RelCategory
    condition: str = ""

# ADR-006 四类关系实例
relations = [
    Relation("is_located_in", "A101", "1F", RelCategory.HIERARCHICAL),
    Relation("is_located_in", "1F", "A栋", RelCategory.HIERARCHICAL),
    Relation("contains", "A101", "EM-001", RelCategory.STRUCTURAL),
    Relation("is_signed_by", "CT2026001", "M-001", RelCategory.IDENTITY_REF),
    Relation("applies_to", "CT2026001", "A101", RelCategory.IDENTITY_REF),
    Relation("activates_occupancy", "CT2026001", "A101", RelCategory.LIFECYCLE_EFFECT, "signed→active"),
    Relation("generates_billing", "CT2026001", "Bill-001", RelCategory.LIFECYCLE_EFFECT, "signed→active"),
]

for r in relations:
    print(f"  {r.source:12} --[{r.predicate}({r.category.value:6})]--> {r.target}")

## 第 2 格：邻接表 + BFS 遍历

In [ ]:
from collections import defaultdict, deque

# 构建邻接表（双向）
graph = defaultdict(list)
for r in relations:
    graph[r.source].append((r.predicate, r.target, r.category))
    graph[r.target].append((r.predicate + "(inv)", r.source, r.category))

def bfs(start, graph, max_depth=5):
    """BFS 遍历，返回可达节点及关系链"""
    visited = {start}
    queue = deque([(start, 0, [start])])
    results = []
    while queue:
        node, depth, path = queue.popleft()
        if depth >= max_depth:
            continue
        for pred, neighbor, cat in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                new_path = path + [f"--[{pred}]-->", neighbor]
                results.append((neighbor, cat, depth + 1, " ".join(new_path)))
                queue.append((neighbor, depth + 1, new_path))
    return results

print("从 A101 出发 BFS 遍历：")
for node, cat, d, path in bfs("A101", graph):
    print(f"  depth={d} {path}")

## 第 3 格：Agent 推理——"A101 为什么不能出租？"

In [ ]:
# Agent 不读外键，沿语义关系推理
def answer_why_not_leasable(unit_id):
    lines = [f"查询：{unit_id} 为什么不能出租？", ""]
    # Step 1: 沿 applies_to 找关联合同
    for r in relations:
        if r.predicate == "applies_to" and r.target == unit_id:
            lines.append(f"1. 找到合同 {r.source} applies_to {unit_id} [身份引用]")
            # Step 2: 检查生命周期影响
            for r2 in relations:
                if r2.source == r.source and r2.category == RelCategory.LIFECYCLE_EFFECT and r2.target == unit_id:
                    lines.append(f"2. {r2.source} 在 {r2.condition} 时 activates_occupancy → {unit_id} 被占用")
                    lines.append(f"3. 结论：需先解除合同占用（Lifecycle Effect），铺位才能出租")
    return "\n".join(lines)

print(answer_why_not_leasable("A101"))

## 第 4 格：可视化——关系图

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
font_name = font_manager.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc").get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 简单点图表示关系
nodes = {"A101", "1F", "A栋", "EM-001", "CT2026001", "M-001", "Bill-001"}
pos = {"A栋": (1, 3), "1F": (1, 2), "A101": (1, 1), "EM-001": (2, 1),
        "M-001": (3, 2), "CT2026001": (2, 2), "Bill-001": (3, 1)}

fig, ax = plt.subplots(figsize=(8, 5))
cat_colors = {RelCategory.IDENTITY_REF: "#2196F3", RelCategory.STRUCTURAL: "#FF9800",
             RelCategory.HIERARCHICAL: "#4CAF50", RelCategory.LIFECYCLE_EFFECT: "#F44336"}

for r in relations:
    x = [pos[r.source][0], pos[r.target][0]]
    y = [pos[r.source][1], pos[r.target][1]]
    ax.plot(x, y, color=cat_colors[r.category], lw=1.5, alpha=0.7)
    mx, my = (x[0]+x[1])/2, (y[0]+y[1])/2
    ax.text(mx, my+0.08, r.predicate, fontsize=7, ha="center", color=cat_colors[r.category])

for n, (x, y) in pos.items():
    ax.plot(x, y, "o", color="#333", ms=10)
    ax.text(x, y-0.2, n, fontsize=8, ha="center")

for cat, color in cat_colors.items():
    ax.plot([], [], color=color, label=cat.value)
ax.legend(fontsize=8, loc="upper right")
ax.set_title("MI CRE 四类语义关系图")
ax.axis("off")
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第10周/d2_relationship_graph.png", dpi=100)
plt.show()
print("关系图已绘制")